# 07 — Training-only Feature Selection

Run 06 first. All selectors, correlation filtering, and ranking below use training patients only. Validation/test patients are not used to fit selection.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.post_05.common import FEATURES_DIR, FIGURES_DIR, ensure_output_dirs, save_csv_new, new_artifact_path
from src.post_05.selection import rank_training_features
ensure_output_dirs()
features=pd.read_csv(FEATURES_DIR/'cycle_features.csv')
splits=pd.read_csv(FEATURES_DIR/'patient_splits.csv')
data=features.merge(splits[['patient_id','split']],on='patient_id',validate='many_to_one')
train=data[data.split.eq('train')].copy()
ranking, selected=rank_training_features(train,max_features=40)
print('Training patients:',train.patient_id.nunique(),'training cycles:',len(train),'selected:',len(selected))
display(ranking.head(30))
save_csv_new(ranking,FEATURES_DIR/'feature_ranking.csv')
save_csv_new(pd.DataFrame({'feature':selected}),FEATURES_DIR/'selected_feature_list.csv')
save_csv_new(data[['patient_id','diagnosis','recording_id','cycle_number','sound_label','split']+selected],FEATURES_DIR/'selected_cycle_features.csv')
plt.figure(figsize=(9,7)); ranking.head(20).sort_values('combined_rank').plot.barh(x='feature',y='random_forest_importance',legend=False); plt.tight_layout(); plt.savefig(new_artifact_path(FIGURES_DIR/'07_training_feature_importance.png'),dpi=160); plt.show()